# **Data Cleaning and Feature Engineering**

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from src.config import get_path
from src.data_loader import (download_wb_indicators, load_wb_nigeria,
                              load_cbn_payments, load_efina_summary, load_competitors)
from src.preprocessing import (build_nigeria_indicators, save_processed,
                                add_yoy_growth, missing_value_report)
from src.viz import apply_project_style
apply_project_style()
print("Setup complete.")

Setup complete.


## Loading and Filtering Nigeria-Only WB Data

I filtered the World Bank data to Nigeria and pivot it to wide format. Wide format is easier to work with for country-level analysis.

In [2]:
df_wb_all = download_wb_indicators(cache=True)
df_wb_ng  = load_wb_nigeria(df_wb_all)

print(f"Nigeria WB data shape: {df_wb_ng.shape}")
print(f"Years covered: {df_wb_ng['year'].min()} – {df_wb_ng['year'].max()}")
display(df_wb_ng.head())

Loading cached World Bank data from C:\Users\Peter\Documents\projects\Jobberman_projects\therbo_consulting\nigeria_dfs_analysis\data\raw\wb_indicators_raw.csv
Nigeria WB data shape: (13, 11)
Years covered: 2011 – 2023


,year,account_female,account_male,account_ownership,gdp_per_capita,gdp_usd,inflation,internet_users,mobile_subscriptions,population_total,urban_population_pct
0,2011,25.990401,33.276893,29.667538,2418.413170,4.144667e+11,10.826137,13.8000,55.530127,171379598.0,52.490352
1,2012,NaN,NaN,NaN,2633.197347,4.639710e+11,12.224241,16.1000,64.005326,176200625.0,53.302115
2,2013,NaN,NaN,NaN,2872.790834,5.201172e+11,8.495518,19.1000,70.282510,181049443.0,54.113086
3,2014,34.022396,54.362481,44.441985,3088.721313,5.741838e+11,8.047411,21.0000,74.751278,185896915.0,54.923244
4,2015,NaN,NaN,NaN,2585.733607,4.930267e+11,9.009435,22.5142,79.104528,190671878.0,55.732568


## Merging All Nigeria Sources

I merged all Nigeria data into one combined DataFrame.

In [ ]:
df_cbn   = load_cbn_payments()
df_efina = load_efina_summary()

df_ng = build_nigeria_indicators(df_wb_ng, df_cbn, df_efina)

print(f"Combined Nigeria DataFrame:")
print(f"  Shape: {df_ng.shape}")
print(f"  Years: {df_ng['year'].dropna().astype(int).tolist()}")
print()
display(df_ng.head(5))

Combined Nigeria DataFrame:
  Shape: (13, 29)
  Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]



,year,account_female,account_male,account_ownership,gdp_per_capita,gdp_usd,inflation,internet_users,mobile_subscriptions,population_total,...,mobile_money_wallets_m,banked_pct,excluded_pct,mobile_money_pct,fx_usd_ngn,nip_value_usd_m,total_digital_vol_m,nip_volume_m_yoy_pct,nip_value_bn_ngn_yoy_pct,total_digital_vol_m_yoy_pct
0,2011,25.990401,33.276893,29.667538,2418.413170,4.144667e+11,10.826137,13.8000,55.530127,171379598.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2012,NaN,NaN,NaN,2633.197347,4.639710e+11,12.224241,16.1000,64.005326,176200625.0,...,NaN,28.6,39.5,2.1,NaN,NaN,0.0,NaN,NaN,NaN
2,2013,NaN,NaN,NaN,2872.790834,5.201172e+11,8.495518,19.1000,70.282510,181049443.0,...,NaN,28.6,39.5,2.1,NaN,NaN,0.0,NaN,NaN,NaN
3,2014,34.022396,54.362481,44.441985,3088.721313,5.741838e+11,8.047411,21.0000,74.751278,185896915.0,...,NaN,36.3,39.5,3.9,NaN,NaN,0.0,NaN,NaN,NaN
4,2015,NaN,NaN,NaN,2585.733607,4.930267e+11,9.009435,22.5142,79.104528,190671878.0,...,3.1,36.3,39.5,3.9,199.0,157788.9,366.0,NaN,NaN,inf


## Derived Features

I computed several derived indicators that are useful for the analysis.

In [4]:
# 3a. Total digital payment volume (sum of all channels)
vol_cols = ['nip_volume_m', 'pos_volume_m', 'mobile_vol_m', 'inet_vol_m']
existing = [c for c in vol_cols if c in df_ng.columns]
if existing:
    df_ng['total_digital_vol_m'] = df_ng[existing].sum(axis=1, skipna=True)
    print(f"Total digital volume 2023: {df_ng[df_ng['year']==2023]['total_digital_vol_m'].values[0]:,.0f}M transactions")

# 3b. YoY growth rates for the key payment metrics
for col in ['nip_volume_m', 'nip_value_bn_ngn', 'pos_volume_m', 'mobile_vol_m']:
    if col in df_ng.columns:
        df_ng = add_yoy_growth(df_ng, col)

# 3c. NIP value in USD millions (approximate, using our FX table)
if 'fx_usd_ngn' in df_ng.columns and 'nip_value_bn_ngn' in df_ng.columns:
    df_ng['nip_value_usd_m'] = (df_ng['nip_value_bn_ngn'] * 1e9 / df_ng['fx_usd_ngn'] / 1e6).round(1)
    print(f"NIP value 2023 (USD): ${df_ng[df_ng['year']==2023]['nip_value_usd_m'].values[0]:,.0f}M")

# 3d. Financial inclusion gap (% excluded)
if 'banked_pct' in df_ng.columns:
    df_ng['inclusion_gap_pct'] = 100 - df_ng['banked_pct']
    print(f"Inclusion gap 2023: {df_ng[df_ng['year']==2023]['inclusion_gap_pct'].dropna().values}")

Total digital volume 2023: 12,582M transactions
NIP value 2023 (USD): $903,333M
Inclusion gap 2023: [55.]


In [5]:
# 3e. CAGR calculations for the key metrics (2015–2023)
from src.stats import cagr

metrics_to_cagr = {
    'NIP Volume (M)': 'nip_volume_m',
    'NIP Value (NGN bn)': 'nip_value_bn_ngn',
    'Mobile Wallets (M)': 'mobile_money_wallets_m',
}

print("CAGR 2015–2023:")
for label, col in metrics_to_cagr.items():
    if col in df_ng.columns:
        row_2015 = df_ng[df_ng['year'] == 2015][col].dropna()
        row_2023 = df_ng[df_ng['year'] == 2023][col].dropna()
        if len(row_2015) and len(row_2023):
            rate = cagr(float(row_2015.iloc[0]), float(row_2023.iloc[0]), 8)
            print(f"  {label:<30} : {rate*100:.1f}%/yr")

CAGR 2015–2023:
  NIP Volume (M)                 : 57.9%/yr
  NIP Value (NGN bn)             : 50.2%/yr
  Mobile Wallets (M)             : 26.2%/yr


## Final Checks Before Saving

In [ ]:
# I ran a final missing-value report on the combined DataFrame.
processed_dir = get_path("data_processed")
missing_value_report(df_ng, label="Combined Nigeria Dataset")


=== Missing Value Report: Combined Nigeria Dataset ===
  Rows: 13
                     column  n_missing  pct_missing   dtype
               account_male          9        69.23 float64
          account_ownership          9        69.23 float64
             account_female          9        69.23 float64
       mobile_vol_m_yoy_pct          5        38.46 float64
       pos_volume_m_yoy_pct          5        38.46 float64
   nip_value_bn_ngn_yoy_pct          5        38.46 float64
       nip_volume_m_yoy_pct          5        38.46 float64
          mobile_val_bn_ngn          4        30.77 float64
           pos_value_bn_ngn          4        30.77 float64
     mobile_money_wallets_m          4        30.77 float64
            inet_val_bn_ngn          4        30.77 float64
                 inet_vol_m          4        30.77 float64
               mobile_vol_m          4        30.77 float64
                 fx_usd_ngn          4        30.77 float64
               pos_volume_m      

,column,n_missing,pct_missing,dtype
0,account_male,9,69.23,float64
1,account_ownership,9,69.23,float64
2,account_female,9,69.23,float64
3,mobile_vol_m_yoy_pct,5,38.46,float64
4,pos_volume_m_yoy_pct,5,38.46,float64
5,nip_value_bn_ngn_yoy_pct,5,38.46,float64
6,nip_volume_m_yoy_pct,5,38.46,float64
7,mobile_val_bn_ngn,4,30.77,float64
8,pos_value_bn_ngn,4,30.77,float64
9,mobile_money_wallets_m,4,30.77,float64


In [7]:
# I save the processed dataset for use in all subsequent notebooks.
save_processed(df_ng, "nigeria_combined.csv", processed_dir)

# Also save the processed competitor data
df_comp = load_competitors()
save_processed(df_comp, "competitors_processed.csv", processed_dir)
print("\nAll processed datasets saved.")

Saved processed dataset: C:\Users\Peter\Documents\projects\Jobberman_projects\therbo_consulting\nigeria_dfs_analysis\data\processed\nigeria_combined.csv
  Shape : (13, 32)
  Cols  : ['year', 'account_female', 'account_male', 'account_ownership', 'gdp_per_capita', 'gdp_usd', 'inflation', 'internet_users', 'mobile_subscriptions', 'population_total', 'urban_population_pct', 'nip_volume_m', 'nip_value_bn_ngn', 'pos_volume_m', 'pos_value_bn_ngn', 'mobile_vol_m', 'mobile_val_bn_ngn', 'inet_vol_m', 'inet_val_bn_ngn', 'mobile_money_wallets_m', 'banked_pct', 'excluded_pct', 'mobile_money_pct', 'fx_usd_ngn', 'nip_value_usd_m', 'total_digital_vol_m', 'nip_volume_m_yoy_pct', 'nip_value_bn_ngn_yoy_pct', 'total_digital_vol_m_yoy_pct', 'pos_volume_m_yoy_pct', 'mobile_vol_m_yoy_pct', 'inclusion_gap_pct']
Saved processed dataset: C:\Users\Peter\Documents\projects\Jobberman_projects\therbo_consulting\nigeria_dfs_analysis\data\processed\competitors_processed.csv
  Shape : (10, 11)
  Cols  : ['company', '